# Human Activity Recognition with Wifi

# Packages import and Data initialization

In [ ]:
import numpy as np
import numpy.random as rnd
import pandas as pd
import torch
import matplotlib.pyplot as plt

from models.DopplerDataset import * 
from models.Architectures import *
from utilities.training import *
from utilities.plotting import *
from utilities.data_processing import *

from time import time 
from torchvision.transforms import Compose, ToTensor, RandomHorizontalFlip, RandomVerticalFlip
from torch.utils.data import DataLoader
from os import listdir, makedirs, path
from glob import glob
from tqdm import tqdm
from pathlib import Path

%load_ext autoreload
%autoreload 2

## Separating training dataset and test dataset

In [ ]:
DEBUG_MODE                 = True

TRAIN_DATASET_PATH  = "doppler_traces/S1*"
TEST_DATASET_PATH   = "doppler_traces/S2*"
DOPPLER_TRACE_SIZE  = 340

LABELS     = ['W', 'E', 'R', 'J', 'L']
ACTIVITIES = ['Walking', 'Empty', 'Running', 'Jumping', 'Sitting']
labels_map = {label:idx for idx, label in enumerate(LABELS)}

create_train_dataset(TRAIN_DATASET_PATH, DOPPLER_TRACE_SIZE, LABELS)
create_test_dataset(TEST_DATASET_PATH, DOPPLER_TRACE_SIZE, LABELS)

In [ ]:
batch_size = 128
augmentation = "hv"  # Options: "h", "v", "hv", None
train_dataset  = DopplerDataset("doppler_traces_train", labels_map, db_conversion=True, normalization=False, augmentation=augmentation)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
train_dataset.getInfo()

test_dataset  = DopplerDataset("doppler_traces_test", labels_map, db_conversion=True, normalization=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)
test_dataset.getInfo()

plot_dataset(3, 3, train_dataset, ACTIVITIES, LABELS)

print("Dimensions of each tensor in the training dataset: ", train_dataset[0][0].shape)
print("Dimensions of each input from the training dataloader: ", next(iter(train_dataloader))[0].shape)
print("Dimensions of each output from the training dataloader: ", next(iter(train_dataloader))[1].shape)

# First model (SHARP with no BN)

## Training of SHARP architecure (no batch normalization)

In [ ]:
sharp_model = SHARP(n_features=len(ACTIVITIES), batchNorm=False)

learning_rate = 1e-4
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=sharp_model.parameters(), lr=learning_rate)

if DEBUG_MODE:
    #### CHECKING DATA TYPES 
    x, y = next(iter(train_dataloader))
    print(x.dtype)
    print(next(sharp_model.parameters()).dtype)
    ### DEVICE
    print(f"Device: {device}")

In [ ]:
epochs = 50
train_loss, test_loss, train_acc, test_acc, _ = train_model(sharp_model, train_dataloader, test_dataloader, epochs, loss_fn, optimizer, device, verbosity=True)

In [ ]:
plot_loss(train_loss, test_loss, train_acc, test_acc, "SHARP model, no batch normalization")

### Metrics

In [ ]:
metrics = compute_metrics("all", sharp_model, test_dataset, LABELS, device, False)

In [ ]:
plot_confusion_matrix(metrics["cm"], LABELS)

# Second model (SHARP with BN)

### Training of SHARP (with BatchNormalization)

In [ ]:
train_dataset  = DopplerDataset("doppler_traces_train", labels_map, db_conversion=True, normalization=False, augmentation="hv")
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
train_dataset.getInfo()

test_dataset  = DopplerDataset("doppler_traces_test", labels_map, db_conversion=True, normalization=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)
test_dataset.getInfo()

plot_dataset(3, 3, train_dataset, ACTIVITIES, LABELS)

In [ ]:
sharp_model = SHARP(n_features=len(ACTIVITIES), batchNorm=True)

learning_rate = 1e-4
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=sharp_model.parameters(), lr=learning_rate, weight_decay=1e-5)
sharp_model
print(f"Device: {device}")

In [ ]:
epochs = 30
train_loss, test_loss, train_acc, test_acc, _ = train_model(sharp_model, train_dataloader, test_dataloader, epochs, loss_fn, optimizer, device, verbosity=True)

In [ ]:
plot_loss(train_loss, test_loss, train_acc, test_acc, "SHARP model, with batch normalization")

## Metrics

In [ ]:
metrics = compute_metrics("all", sharp_model, test_dataset, LABELS, device, False)

In [ ]:
plot_confusion_matrix(metrics["cm"], LABELS)

In [ ]:
plot_f1_score(metrics["precision"], metrics["recall"], metrics["f1"], LABELS)
print(f"Mean f1-score: {metrics["f1"].mean():.4f}")

### Save best model

In [ ]:
save_best_F1_model(sharp_model, metrics) # based on F1-score

# Third model:  ResNet-18 implementation

In [ ]:
from torchvision.models import resnet18

resnet_model = resnet18(weights=None)

### Modify the model to adapt to our case

In [ ]:
# Modify input layer to accept single-channel input (1 channel instead of 3)
resnet_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# Modify output layer to match the number of classes in the dataset
num_classes = len(ACTIVITIES)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

if DEBUG_MODE:
    print(f"Modified ResNet18 model for {num_classes} classes:")
    print(resnet_model)

# # Initialize weights for the modified layers
# nn.init.kaiming_normal_(resnet_model.conv1.weight, mode='fan_out', nonlinearity='relu')
# nn.init.kaiming_normal_(resnet_model.fc.weight)
# nn.init.constant_(resnet_model.fc.bias, 0)

In [ ]:
learning_rate = 1e-4
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=resnet_model.parameters(), lr=learning_rate)

if DEBUG_MODE:
    #### CHECKING DATA TYPES 
    x, y = next(iter(train_dataloader))
    print(x.dtype)
    print(next(resnet_model.parameters()).dtype)
    ### DEVICE
    print(f"Device: {device}")

### Training

In [ ]:
epochs = 30
train_loss, test_loss, train_acc, test_acc, _ = train_model(resnet_model, train_dataloader, test_dataloader, epochs, loss_fn, optimizer, device, verbosity=True)

In [ ]:
plot_loss(train_loss, test_loss, train_acc, test_acc, "ResNet-18 model")

## Metrics

In [ ]:
metrics = compute_metrics("all", resnet_model, test_dataset, LABELS, device, False)
plot_confusion_matrix(metrics["cm"], LABELS)

In [ ]:
plot_f1_score(metrics["precision"], metrics["recall"], metrics["f1"], LABELS)
print(f"Mean f1-score: {metrics["f1"].mean():.4f}")

# HYPERPARAMETER AUTO-TUNING

In [ ]:
import optuna
import gc

## SHARP without BN

In [ ]:
best_accuracy = 0.0
best_model_state = None
batchNorm = False

test_dataset  = DopplerDataset("doppler_traces_test", labels_map, db_conversion=True, normalization=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

augmentation = "hv"  # Options: "h", "v", "hv", None

def objective(trial):
    global best_accuracy, best_model_state

    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])

    train_dataset  = DopplerDataset("doppler_traces_train", labels_map, db_conversion=True, normalization=False, augmentation=augmentation)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    sharp_model = SHARP(n_features=len(ACTIVITIES), batchNorm=batchNorm)

    device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    loss_fn   = nn.CrossEntropyLoss()
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    optimizer = torch.optim.Adam(params=sharp_model.parameters(), lr=learning_rate)

    sharp_model.to(device)

    epochs = 50
    _, _, _, test_acc, _ = train_model(sharp_model, train_dataloader, test_dataloader, epochs,
                                       loss_fn, optimizer, device, verbosity=False, early_stopping=True, patience=15)
    score = max(test_acc)
    if score > best_accuracy:
        best_accuracy = score
        best_model_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in sharp_model.state_dict().items()
        }

    # Free GPU memory
    del sharp_model, train_dataset, train_dataloader, optimizer, loss_fn
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    
    return score   # Return the final test accuracy

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

# Print the best hyperparameters found
trial = study.best_trial
print("Best hyperparameters: ", trial.params)
print("Best accuracy: ", trial.value)

### Compute metrics and save best model

In [ ]:
sharp_model_noBN = SHARP(n_features=len(ACTIVITIES), batchNorm=batchNorm)
sharp_model_noBN.load_state_dict(best_model_state)
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
sharp_model_noBN.to(device)

# Save the best model state dictionary to a file
Path("best_models").mkdir(parents=True, exist_ok=True)
# Check if there already exsts a model with the same name saved in the best_models directory, if so, append a number to the filename
model_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN.pth"
counter = 1
while path.exists(model_filename):
    model_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_{counter}.pth"
    counter += 1
torch.save(best_model_state, model_filename)

# Save the hyperparameters of the best model as a npz file
hyperparams = {"learning_rate": trial.params["learning_rate"],
               "BN": batchNorm,
               "batch_size": trial.params["batch_size"],
               "optimizer": "Adam",
               "loss_fn": "CrossEntropyLoss",
               "augmentation": augmentation}
hyperparams_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_hyperparameters.npz"
counter = 1
while path.exists(hyperparams_filename):
    hyperparams_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_hyperparameters_{counter}.npz"
    counter += 1
np.savez(hyperparams_filename, **hyperparams)

# Compute and save metrics, plot confusion matrix for the best model
metrics = compute_metrics("all", sharp_model_noBN, test_dataset, LABELS, device, False)
metrics_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_metrics.npz"
counter = 1
while path.exists(metrics_filename):
    metrics_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_metrics_{counter}.npz"
    counter += 1
np.savez(metrics_filename, **metrics)
plot_confusion_matrix(metrics["cm"], LABELS)

In [ ]:
plot_f1_score(metrics["precision"], metrics["recall"], metrics["f1"], LABELS)
print(f"Mean f1-score: {metrics["f1"].mean():.4f}")

## SHARP with BN

In [ ]:
best_accuracy = 0.0
best_model_state = None
batchNorm = True

test_dataset  = DopplerDataset("doppler_traces_test", labels_map, db_conversion=True, normalization=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

augmentation = "hv"  # Options: "h", "v", "hv", None

def objective(trial):
    global best_accuracy, best_model_state

    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])

    train_dataset  = DopplerDataset("doppler_traces_train", labels_map, db_conversion=True, normalization=False, augmentation=augmentation)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    sharp_model = SHARP(n_features=len(ACTIVITIES), batchNorm=batchNorm)

    device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    loss_fn   = nn.CrossEntropyLoss()
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    optimizer = torch.optim.Adam(params=sharp_model.parameters(), lr=learning_rate)

    sharp_model.to(device)

    epochs = 50
    _, _, _, test_acc, _ = train_model(sharp_model, train_dataloader, test_dataloader, epochs,
                                       loss_fn, optimizer, device, verbosity=False, early_stopping=True, patience=15)
    score = max(test_acc)
    if score > best_accuracy:
        best_accuracy = score
        best_model_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in sharp_model.state_dict().items()
        }

    # Free GPU memory
    del sharp_model, train_dataset, train_dataloader, optimizer, loss_fn
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    
    return score   # Return the final test accuracy

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

# Print the best hyperparameters found
trial = study.best_trial
print("Best hyperparameters: ", trial.params)
print("Best accuracy: ", trial.value)

### Compute metrics and save best model

In [ ]:
sharp_model_withBN = SHARP(n_features=len(ACTIVITIES), batchNorm=batchNorm)
sharp_model_withBN.load_state_dict(best_model_state)
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
sharp_model_withBN.to(device)

# Save the best model state dictionary to a file
Path("best_models").mkdir(parents=True, exist_ok=True)
# Check if there already exsts a model with the same name saved in the best_models directory, if so, append a number to the filename
model_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN.pth"
counter = 1
while path.exists(model_filename):
    model_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_{counter}.pth"
    counter += 1
torch.save(best_model_state, model_filename)

# Save the hyperparameters of the best model as a npz file
hyperparams = {"learning_rate": trial.params["learning_rate"],
               "BN": batchNorm,
               "batch_size": trial.params["batch_size"],
               "optimizer": "Adam",
               "loss_fn": "CrossEntropyLoss",
               "augmentation": augmentation}
hyperparams_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_hyperparameters.npz"
counter = 1
while path.exists(hyperparams_filename):
    hyperparams_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_hyperparameters_{counter}.npz"
    counter += 1
np.savez(hyperparams_filename, **hyperparams)

# Compute and save metrics, plot confusion matrix for the best model
metrics = compute_metrics("all", sharp_model_withBN, test_dataset, LABELS, device, False)
metrics_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_metrics.npz"
counter = 1
while path.exists(metrics_filename):
    metrics_filename = f"best_models/sharp_{'with' if batchNorm else 'no'}BN_metrics_{counter}.npz"
    counter += 1
np.savez(metrics_filename, **metrics)
plot_confusion_matrix(metrics["cm"], LABELS)

In [ ]:
plot_f1_score(metrics["precision"], metrics["recall"], metrics["f1"], LABELS)
print(f"Mean f1-score: {metrics["f1"].mean():.4f}")

In [ ]:
# Plot hyperparameters of the study
if DEBUG_MODE:
    print("Study trials: ", study.trials)

    completed_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
    ]

    trial_numbers = [trial.number for trial in completed_trials]
    learning_rates = [trial.params["learning_rate"] for trial in completed_trials]
    scores = [trial.value for trial in completed_trials]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(trial_numbers, learning_rates, marker="o")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Trial number")
    axes[0].set_ylabel("Learning rate")
    axes[0].set_title("Learning rate by trial")
    axes[0].grid(True)

    axes[1].plot(trial_numbers, scores, marker="o")
    axes[1].set_xlabel("Trial number")
    axes[1].set_ylabel("Test accuracy")
    axes[1].set_title("Objective value by trial")
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


## ResNet-18 

In [ ]:
from torchvision.models import resnet18

In [ ]:
best_accuracy = 0.0
best_model_state = None

def objective(trial):
    global best_accuracy, best_model_state
    
    resnet_model = resnet18(weights=None)
    # Modify input layer to accept single-channel input (1 channel instead of 3)
    resnet_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    # Modify output layer to match the number of classes in the dataset
    num_classes = len(ACTIVITIES)
    resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

    device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    if device == torch.device("cpu"):
        print("Warning: Training on CPU may be slow. Consider using a GPU, if available.")
    loss_fn   = nn.CrossEntropyLoss()
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    optimizer = torch.optim.Adam(params=resnet_model.parameters(), lr=learning_rate)

    # Training
    epochs = 30
    _, _, _, test_acc, _ = train_model(resnet_model, train_dataloader, test_dataloader, epochs, loss_fn, optimizer, device, 
                                       verbosity=False, early_stopping=True, patience=10)
    score = max(test_acc)
    if score > best_accuracy:
        best_accuracy = score
        best_model_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in resnet_model.state_dict().items()
        }
    return score   # Return the final test accuracy

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=12)

# Print the best hyperparameters found
trial = study.best_trial
print("\nBest hyperparameters: ", trial.params)
print("Best accuracy: ", trial.value)

### Compute metrics and save best model

In [ ]:
resnet_model = resnet18(weights=None)
# Modify input layer to accept single-channel input (1 channel instead of 3)
resnet_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
# Modify output layer to match the number of classes in the dataset
num_classes = len(ACTIVITIES)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

resnet_model.load_state_dict(best_model_state)
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
resnet_model.to(device)

# Save the best model state dictionary to a file
torch.save(best_model_state, f"best_models/resnet.pth")
# Save the hyperparameters of the best model as a npz file
hyperparams = {"learning_rate": trial.params["learning_rate"],
               "batch_size": batch_size,
               "optimizer": "Adam",
               "loss_fn": "CrossEntropyLoss",
               "augmentation": augmentation}
np.savez(f"best_models/resnet_hyperparameters.npz", **hyperparams)

# Compute and save metrics and plot confusion matrix for the best model
metrics = compute_metrics("all", resnet_model, test_dataset, LABELS, device, False)
np.savez(f"best_models/resnet_metrics.npz", **metrics)
plot_confusion_matrix(metrics["cm"], LABELS)

In [ ]:
plot_f1_score(metrics["precision"], metrics["recall"], metrics["f1"], LABELS)
print(f"Mean f1-score: {metrics["f1"].mean():.4f}")

# Generalization to different subjects

IDEA: $\newline$
-) Training with dataset of person 1 $\newline$
-) Testing with dataset of person 1 (benchmark), 2, 3 

In [ ]:
TRAIN_DATASET_PATH  = "doppler_traces/S1*"
TEST_DATASET = [2, 3, 7]
DATASET_NAME = "subject_generalization"
DOPPLER_TRACE_SIZE  = 340

LABELS     = ['W', 'E', 'R', 'J', 'L']
ACTIVITIES = ['Walking', 'Empty', 'Running', 'Jumping', 'Sitting']
labels_map = {label:idx for idx, label in enumerate(LABELS)}

for s in TEST_DATASET: 
    create_test_dataset(f"doppler_traces/S{s}*", DOPPLER_TRACE_SIZE, LABELS, DATASET_NAME+f"_S{s}_")

In [ ]:
# train_dataset = DopplerDataset(f"doppler_traces_{DATASET_NAME}_train", labels_map, db_conversion=True, normalization=False, augmentation="hv")
# train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# train_dataset.getInfo()

test_datasets = {}
for s in TEST_DATASET:
    test_dataset = DopplerDataset(f"doppler_traces_{DATASET_NAME}_S{s}_test", labels_map, db_conversion=True, normalization=False)
    test_datasets[s] = test_dataset

In [ ]:
def general_evaluation(model, datasets, device, metrics_names, debug):
    evaluation_dict = {}
    person_key = {2:1, 3:2, 7:3} # maps dataset idx to the person performing the activities

    for s, test_dataset in datasets.items():
        if debug: print(f"Evaluating {person_key[s]}-th person...")

        data_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
        model.to(device)
        metrics = {}

        loss, accuracy = test_loop(model, data_loader, device, debug, fusion="soft")
        metrics["loss"] = loss.cpu().item()
        metrics["accuracy"] = accuracy
    
        computed_metrics = compute_metrics(metrics_names, model, test_dataset, LABELS, device, debug)
        for metrics_name, computed_metric in computed_metrics.items():
            metrics[metrics_name] = computed_metric

        evaluation_dict[person_key[s]] = metrics
        if debug: print(f"\n{"="*20}\n")

    return evaluation_dict

def displayResults(evaluation_dict):
    new_eval_dict = {}
    for person, metrics in evaluation_dict.items():
        new_metrics = {}
        if "cm" in metrics.keys():
            new_metrics = {name_metric: metric for name_metric, metric in metrics.items() if name_metric != "cm"}
        new_eval_dict[person] = new_metrics

    df = pd.DataFrame(new_eval_dict)
    #df.loc["f1"] = df.loc["f1"].apply(np.mean)

    def format_value(x):
        if isinstance(x, (int, float, np.number)):
            return f"{x:.3f}"
        return x

    display(df.style.format(format_value))


## Evalutation with SHARP (no BN)

In [ ]:
batchNorm = False
sharp_model1 = SHARP(n_features=len(ACTIVITIES), batchNorm=batchNorm)
device = \
    torch.device("mps") if torch.backends.mps.is_available() else \
    torch.device("cuda") if torch.cuda.is_available() else \
    torch.device("cpu")

best_sharp_noBN_weight = torch.load(
    "best_models/sharp_noBN.pth",
    map_location=device,
    weights_only=True
)

sharp_model1.load_state_dict(best_sharp_noBN_weight)
sharp_model1.to(device)
evaluation = general_evaluation(sharp_model1, test_datasets, device, "all", False)

In [ ]:
displayResults(evaluation)

In [ ]:
for person, metrics in evaluation.items():
    plot_confusion_matrix(metrics["cm"], LABELS, title=f"Matrice di Confusione del Modello per la persona {person}")

## Evaluation with SHARP (with BN)

In [ ]:
batchNorm = True
sharp_model2 = SHARP(n_features=len(ACTIVITIES), batchNorm=batchNorm)
device = \
    torch.device("mps") if torch.backends.mps.is_available() else \
    torch.device("cuda") if torch.cuda.is_available() else \
    torch.device("cpu")

best_sharp_withBN_weight = torch.load(
    "best_models/sharp_withBN.pth",
    map_location="cpu",
    weights_only=False
)

sharp_model2.load_state_dict(best_sharp_withBN_weight)
sharp_model2.to(device)
evaluation = general_evaluation(sharp_model2, test_datasets, device, "all", False)

In [ ]:
displayResults(evaluation)

In [ ]:
for person, metrics in evaluation.items():
    plot_confusion_matrix(metrics["cm"], LABELS, title=f"Matrice di Confusione del Modello per la persona {person}")

## Evaluation with ResNet

In [ ]:
resnet_model = resnet18(weights=None)
# Modify input layer to accept single-channel input (1 channel instead of 3)
resnet_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
# Modify output layer to match the number of classes in the dataset
num_classes = len(ACTIVITIES)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)
device = \
    torch.device("mps") if torch.backends.mps.is_available() else \
    torch.device("cuda") if torch.cuda.is_available() else \
    torch.device("cpu")

best_resnet_weight = torch.load(
    "best_models/resnet.pth",
    weights_only=True
)

resnet_model.load_state_dict(best_resnet_weight)
resnet_model.to(device)
evaluation = general_evaluation(resnet_model, test_datasets, device, "all", False)

In [ ]:
displayResults(evaluation)

In [ ]:
for person, metrics in evaluation.items():
    plot_confusion_matrix(metrics["cm"], LABELS, title=f"Matrice di Confusione del Modello per la persona {person}")

Try also different batch size